# 04 · Обучение MobileNetV3-Small
Готовая архитектура torchvision и веса ImageNet; последний слой заменён бинарным выходом.
Дообучаются все параметры. Используются реальные кропы HierText.
Train: контраст, яркость, размытие, уменьшение разрешения, JPEG и наклон ±7° с вероятностью 50%.
Метка 0/180 создаётся после аугментаций. Dev содержит обе ориентации каждого кропа.
Лучшие веса выбираются по dev Brier и сохраняются в best.pt. last.pt нужен для продолжения.
Проверь RUN_DIR; существующие результаты не перезаписываются новым запуском.


In [ ]:
from pathlib import Path


def find_project_root(start=None):
    from pathlib import Path
    import os

    current = Path(start or Path.cwd()).resolve()
    explicit = os.environ.get("AVITO_PROJECT_ROOT")
    candidates = (
        [Path(explicit).expanduser().resolve()] if explicit else [current, *current.parents]
    )
    for candidate in candidates:
        marker = (candidate / "avito_project.json").is_file()
        layout = (candidate / "notebooks" / "_shared.ipynb").is_file() and (
            candidate / "requirements.txt"
        ).is_file()
        if marker or layout:
            return candidate
    raise RuntimeError(
        f"Не найден корень Avito_test. Текущая папка: {current}. "
        "Перенеси notebooks/ целиком, requirements.txt и avito_project.json в папку проекта. "
        "На VM выполни в отдельной ячейке: %cd /home/slomauh/ml/Avito_test . "
        "Для другого расположения задай переменную окружения AVITO_PROJECT_ROOT. "
        + (f"Сейчас AVITO_PROJECT_ROOT={explicit!r}; проверь этот путь." if explicit else "")
    )


ROOT = find_project_root()
get_ipython().run_line_magic("run", str(ROOT / "notebooks" / "_shared.ipynb"))
print("Проект:", ROOT)


In [ ]:
MANIFEST = ROOT / "data/processed/hiertext_full/manifest.csv"
RUN_DIR = ROOT / "runs/balanced_A_rot7_v1"
RESUME = False
STOP_AFTER_EPOCH = None  # например 3: сохранить эпоху и остановиться; CFG epochs не меняется
MAX_TRAIN_CROPS = 200000
CFG = dict(
    seed=42,
    height=48,
    width=192,
    batch_size=128,
    epochs=10,
    lr=3e-4,
    weight_decay=1e-4,
    patience=None,
    amp=True,
    pretrained=True,
    cpu_threads=4,
    allow_cpu=False,
    rotation_degrees=7.0,
    rotation_probability=0.5,
)


In [ ]:
manifest = load_manifest(MANIFEST)
train_frame = manifest[manifest.split == "train"].sort_values("crop_id").copy()
dev_frame = manifest[manifest.split == "dev"].copy()
if MAX_TRAIN_CROPS is not None and len(train_frame) > MAX_TRAIN_CROPS:
    train_frame = train_frame.sample(MAX_TRAIN_CROPS, random_state=CFG["seed"])
train_frame = train_frame.sort_values("crop_id").reset_index(drop=True)
if len(train_frame) < 5 or dev_frame.empty:
    raise ValueError("Недостаточно train/dev")
data_identity = dict(
    manifest_sha256=sha256_file(MANIFEST),
    max_train_crops=MAX_TRAIN_CROPS,
    epoch_size=len(train_frame),
    selected_crop_ids_sha256=hashlib.sha256("\n".join(train_frame.crop_id).encode()).hexdigest(),
)
print("Папка запуска:", RUN_DIR)
print(
    "Примеров/эпоху:",
    len(train_frame),
    "Шагов/эпоху:",
    math.ceil(len(train_frame) / CFG["batch_size"]),
)
display(train_frame.groupby("script").size().rename("crops").to_frame())


In [ ]:
def rng_state():
    n = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": [n[0], n[1].tolist(), n[2], n[3], n[4]],
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    }


def restore_rng(state):
    random.setstate(state["python"])
    n = state["numpy"]
    np.random.set_state((n[0], np.asarray(n[1], dtype=np.uint32), n[2], n[3], n[4]))
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state["cuda"]:
        torch.cuda.set_rng_state_all(state["cuda"])


def atomic_torch_save(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


def train_experiment(
    train_frame, dev_frame, cfg, run_dir, data_identity, resume=False, stop_after_epoch=None
):
    run_dir = Path(run_dir)
    if run_dir.exists() and not resume:
        raise FileExistsError("RUN_DIR уже существует. Выбери новое имя или RESUME=True.")
    if resume and not (run_dir / "last.pt").exists():
        raise FileNotFoundError("Для RESUME нужен last.pt")
    run_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(cfg["seed"])
    torch.set_num_threads(cfg["cpu_threads"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda" and not cfg["allow_cpu"]:
        raise RuntimeError("GPU не обнаружена. Для короткой проверки можно включить allow_cpu.")
    model = build_model(pretrained=cfg["pretrained"] and not resume).to(device)
    initial_hash = weights_digest(model)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
    scaler = torch.amp.GradScaler("cuda", enabled=cfg["amp"] and device.type == "cuda")
    loss_fn = nn.BCEWithLogitsLoss()
    start_epoch, best_brier, stale, history = 0, float("inf"), 0, []
    if resume:
        state = torch.load(run_dir / "last.pt", map_location="cpu", weights_only=True)
        # Совместимость с ранее сохранёнными запусками на тех же реальных данных.
        previous = state["data_identity"]
        if (
            state["config"] != cfg
            or previous.get("mode", "A") != "A"
            or any(previous.get(k) != v for k, v in data_identity.items())
        ):
            raise ValueError(
                "Config или данные изменились: для нового эксперимента нужен новый RUN_DIR."
            )
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        start_epoch, best_brier, stale = state["epoch"] + 1, state["best_brier"], state["stale"]
        history = state["history"]
        # Восстановление RNG сохраняет последовательность аугментаций после resume.
        restore_rng(state["rng"])
    else:
        save_json(run_dir / "initial_weights.json", {"sha256": initial_hash})
        save_json(run_dir / "config.json", cfg)
        save_json(run_dir / "data_identity.json", data_identity)
        save_json(run_dir / "environment.json", environment())
        import importlib.metadata

        frozen = sorted(
            {
                f'{d.metadata["Name"]}=={d.version}'
                for d in importlib.metadata.distributions()
                if d.metadata["Name"]
            }
        )
        (run_dir / "requirements_frozen.txt").write_text("\n".join(frozen) + "\n", encoding="utf-8")
        train_frame[["crop_id", "source", "path"]].to_csv(
            run_dir / "training_selection.csv", index=False
        )
    train_loader = make_loader(train_frame, cfg, training=True)
    dev_loader = make_loader(dev_frame, cfg)
    print("Устройство:", device, "| Параметров:", sum(p.numel() for p in model.parameters()))
    print(
        "Train upright:", len(train_frame), "| Dev с двумя ориентациями:", len(dev_loader.dataset)
    )
    for epoch in range(start_epoch, cfg["epochs"]):
        if cfg.get("patience") is not None and stale >= cfg["patience"]:
            print("Early stopping уже достигнут.")
            break
        model.train()
        total_loss, seen = 0.0, 0
        start = time.perf_counter()
        for x, y, _ in tqdm(train_loader, desc=f'Эпоха {epoch+1}/{cfg["epochs"]}'):
            x = x.to(device, non_blocking=True)
            y = y.to(device, dtype=torch.float32, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type="cuda", dtype=torch.float16, enabled=scaler.is_enabled()
            ):
                z = model(x).flatten()
                loss = loss_fn(z, y)
            if not torch.isfinite(loss):
                raise FloatingPointError("Loss стал NaN/Inf; проверь данные и learning rate")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.detach()) * len(y)
            seen += len(y)
        # Калибровка и submission тоже FP32: одинаковая численная схема.
        dev_pred = predict_loader(model, dev_loader, device, amp=False)
        scores = metrics(dev_pred.y, probabilities(dev_pred))
        row = {
            "epoch": epoch + 1,
            "train_bce": total_loss / seen,
            "dev_brier": scores["brier"],
            "dev_score": scores["score"],
            "dev_accuracy": scores["accuracy"],
            "steps": len(train_loader),
            "examples": seen,
            "seconds": time.perf_counter() - start,
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(row)
        if scores["brier"] < best_brier:
            best_brier, stale = scores["brier"], 0
            atomic_torch_save(
                {
                    "model": model.state_dict(),
                    "config": cfg,
                    "data_identity": data_identity,
                    "epoch": epoch,
                    "dev_brier": best_brier,
                },
                run_dir / "best.pt",
            )
            dev_pred.to_csv(run_dir / "best_dev_predictions.csv", index=False)
        else:
            stale += 1
        # Сохраняем полное состояние только после завершённой эпохи.
        scheduler.step()
        atomic_torch_save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "epoch": epoch,
                "best_brier": best_brier,
                "stale": stale,
                "history": history,
                "rng": rng_state(),
                "config": cfg,
                "data_identity": data_identity,
            },
            run_dir / "last.pt",
        )
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        if stop_after_epoch is not None and epoch + 1 >= stop_after_epoch:
            break  # Планировщик сохраняет полный горизонт для продолжения.
    return pd.DataFrame(history)


In [ ]:
history = train_experiment(
    train_frame,
    dev_frame,
    CFG,
    RUN_DIR,
    data_identity,
    resume=RESUME,
    stop_after_epoch=STOP_AFTER_EPOCH,
)
display(history)


In [ ]:
import matplotlib.pyplot as plt

history = pd.read_csv(RUN_DIR / "history.csv")
history.plot(x="epoch", y="dev_score", marker="o", grid=True, title="Dev: 1 − Brier")
plt.show()
print("Лучшая модель:", RUN_DIR / "best.pt")


При разрыве соединения веса сохраняются после каждой законченной эпохи.
Для продолжения: тот же CFG, данные и RUN_DIR, `RESUME=True`.
Незаконченная эпоха будет запущена заново с последнего checkpoint.
Если меняешь разрешение, данные, число эпох или другие параметры — новый RUN_DIR.

Не держи одновременно несколько GPU-ядер с моделями: они занимают видеопамять.
После обучения закрой ядро 04 и открой 05 для калибровки/предсказания.
